# 08 · Muestreo presencia–fondo

Paso 29 y preparación del paso 30. U significa **no etiquetado**, nunca ausencia confirmada. Las realizaciones se generan únicamente dentro de cada entrenamiento externo/interno. Los positivos retenidos no intervienen en la exclusión del fondo.

In [1]:
from pathlib import Path
import sys
import importlib
import pandas as pd
from IPython.display import display
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / 'src/geoau/evaluation.py').is_file())
if str(ROOT / 'src') not in sys.path: sys.path.insert(0, str(ROOT / 'src'))
from geoau import evaluation as ev
importlib.reload(ev)

RUN = ev.current_run(ROOT)
samples = ev.build_samples(ROOT, RUN)
display(samples)

Muestras P/U: outer_00


Muestras P/U: outer_00_inner_00


Muestras P/U: outer_00_inner_01


Muestras P/U: outer_00_inner_02


Muestras P/U: outer_01


...

Muestras P/U: outer_04


Muestras P/U: outer_04_inner_00


Muestras P/U: outer_04_inner_01


Muestras P/U: outer_04_inner_02


,sample_id,split_id,level,ratio_requested,realization,seed,n_P,n_U,available_U,requested_U,ratio_realized,pool_capped,mode
0,outer_00_ratio1_rep00,outer_00,outer,1,0,17540843559931103717,378,378,284617,378,1.0,False,diagnostic
1,outer_00_ratio1_rep01,outer_00,outer,1,1,17196037450305951916,378,378,284617,378,1.0,False,diagnostic
2,outer_00_ratio1_rep02,outer_00,outer,1,2,10068290524575453318,378,378,284617,378,1.0,False,diagnostic
3,outer_00_ratio3_rep00,outer_00,outer,3,0,687700831304447256,378,1134,284617,1134,3.0,False,diagnostic
4,outer_00_ratio3_rep01,outer_00,outer,3,1,7109537236180380771,378,1134,284617,1134,3.0,False,diagnostic
...,...,...,...,...,...,...,...,...,...,...,...,...,...
175,outer_04_inner_02_ratio3_rep01,outer_04_inner_02,inner,3,1,15620667963746313437,207,621,167641,621,3.0,False,diagnostic
176,outer_04_inner_02_ratio3_rep02,outer_04_inner_02,inner,3,2,15354475952246053932,207,621,167641,621,3.0,False,diagnostic
177,outer_04_inner_02_ratio10_rep00,outer_04_inner_02,inner,10,0,8713374109143700863,207,2070,167641,2070,10.0,False,diagnostic
178,outer_04_inner_02_ratio10_rep01,outer_04_inner_02,inner,10,1,16959224804719395779,207,2070,167641,2070,10.0,False,diagnostic


## Ratios y probabilidades de inclusión
Se ensayan 1/3/10 U por P y tres realizaciones. El buffer de 250 m se aplica conservadoramente entre huellas de celdas de 1 km: no representa una precisión demostrada de las coordenadas. Es distinto de los 5 km entre entrenamiento y prueba.

El muestreo por bloques guarda probabilidades de inclusión, pesos inversos y pesos de área. Cuando hay menos muestras que bloques se seleccionan primero bloques al azar. Estos pesos no son `class_weight` ni estimaciones de prevalencia.

In [2]:
display(samples.groupby(['level', 'ratio_requested']).agg(
    designs=('sample_id','size'), min_P=('n_P','min'), max_P=('n_P','max'),
    min_U=('n_U','min'), capped=('pool_capped','sum')))
example = samples.sample_id.iloc[0]
display(pd.read_parquet(RUN / f'samples/{example}.parquet').head())
display(pd.read_parquet(RUN / f'samples/{example}_allocation.parquet').head())

designs  min_P  max_P  min_U  capped
level ratio_requested                                      
inner 1                     45    170    462    170       0
      3                     45    170    462    510       0
      10                    45    170    462   1700       0
outer 1                     15    378    533    378       0
      3                     15    378    533   1134       0
      10                    15    378    533   3780       0

,cell_id,block_id,unit_id,land_area_m2,sample_role,sample_class,inclusion_probability,inverse_inclusion_weight,area_weight_m2,mode,training_allowed
0,es_pen_utm30_1km_v1_r0019_c0150,b0_3,b0_3,1000000.0,P_candidate_proxy,1,1.0,1.0,NaN,diagnostic,False
1,es_pen_utm30_1km_v1_r0025_c0124,b0_2,b0_2,1000000.0,P_candidate_proxy,1,1.0,1.0,NaN,diagnostic,False
2,es_pen_utm30_1km_v1_r0025_c0148,b0_2,b0_2,1000000.0,P_candidate_proxy,1,1.0,1.0,NaN,diagnostic,False
3,es_pen_utm30_1km_v1_r0033_c0242,b0_4,b0_4,1000000.0,P_candidate_proxy,1,1.0,1.0,NaN,diagnostic,False
4,es_pen_utm30_1km_v1_r0033_c0244,b0_4,b0_4,1000000.0,P_candidate_proxy,1,1.0,1.0,NaN,diagnostic,False


,block_id,pool_cells,sampled_cells,stratum_selection_probability,cell_inclusion_probability
0,b0_10,55,1,1.0,0.018182
1,b0_2,771,1,1.0,0.001297
2,b0_3,1600,2,1.0,0.001250
3,b0_4,552,1,1.0,0.001812
4,b0_5,772,1,1.0,0.001295


La evaluación usa todas las celdas elegibles `role=test` de la misma membresía, con independencia del ratio y de la realización. No se ha inventado una capa de esfuerzo de observación: esa alternativa exige información verificable.